# 📊 Summaries & Keywords with Gemini

Condense long texts and pull out keywords — one document at a time, or hundreds of rows
in a spreadsheet.

**Run the steps in order, from top to bottom.** Each one has a ▶️ button on its left.

> ⚠️ **Before you upload research material, read Step 1's privacy notice.** On the free
> Gemini tier, Google's terms allow human reviewers to read what you send.

## Step 1: Setup (run this first) ⚙️

Click ▶️ to install the software this notebook needs. It takes about a minute.

In [ ]:
# ============================================
# STEP 1 — SETUP
# ============================================
# Versions are pinned so a new release can't break this notebook mid-workshop.
%pip install -q "google-genai>=2.14,<3" "openpyxl>=3.1,<4" "ipywidgets>=8.1,<9" "ipyfilechooser>=0.6,<1"

# The shared helper module lives in the repository, so a fix reaches all three
# notebooks at once instead of being copy-pasted into each of them.
!wget -q -O zmo_common.py https://raw.githubusercontent.com/fmadore/zmo-ai-pipelines/main/zmo_common.py

import importlib
import json
import os
import re
import shutil
from pathlib import Path

if not os.path.exists('zmo_common.py') or os.path.getsize('zmo_common.py') < 2000:
    raise SystemExit(
        "❌ Could not download the shared helper file (zmo_common.py).\n"
        "   Check your internet connection, then run this cell again."
    )

import zmo_common
importlib.reload(zmo_common)
import zmo_common as zc

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ============================================
# SUPPORTED FORMATS
# ============================================
TEXT_EXTENSIONS = {'.txt', '.md'}
SHEET_EXTENSIONS = {'.xlsx', '.xls'}
ALL_EXTENSIONS = TEXT_EXTENSIONS | SHEET_EXTENSIONS


def document_icon(path):
    return "📊" if Path(path).suffix.lower() in SHEET_EXTENSIONS else "📝"


# ============================================
# FOLDERS
# ============================================
FOLDERS = {
    'input': 'input_files',
    'results': 'results',
    'prompts': 'prompts',
}
for folder in FOLDERS.values():
    os.makedirs(folder, exist_ok=True)

drive = zc.DriveHelper('Colab_Summaries')

# ============================================
# PROMPT TEMPLATE
# ============================================
PROMPT_CONTENT = {
    "summary_prompt.md": """# Summary and Keywords Generation Prompt

Read the text below and produce two things.

## summary
A few concise sentences that let a reader grasp the main content. No introduction, no
commentary about the text, no markdown formatting.

## keywords
Between 5 and 10 keywords or short key phrases covering the main topics and themes.

Write in the same language as the source text unless told otherwise.

---

**Text:**
{text}
""",
}

# Asking for JSON in a fixed shape is far more dependable than asking for a
# "Keywords:" line and parsing it back out of prose.
SUMMARY_SCHEMA = {
    "type": "object",
    "properties": {
        "summary": {"type": "string"},
        "keywords": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["summary", "keywords"],
}

for filename, content in PROMPT_CONTENT.items():
    Path(FOLDERS['prompts'], filename).write_text(content, encoding='utf-8')

print("✅ Setup complete!\n")
zc.environment_report()
print("\n📁 Folders created:")
print("   ├── 📂 input_files/   your texts and spreadsheets")
print("   ├── 📂 results/       summaries")
print("   └── 📂 prompts/       editable instruction template")
print("\n📝 Text files:", ", ".join(sorted(TEXT_EXTENSIONS)))
print("📊 Spreadsheets:", ", ".join(sorted(SHEET_EXTENSIONS)))

display(HTML(zc.privacy_notice("the texts you upload and the summaries it produces")))

## Step 2: Connect your Gemini API key 🔑

The safest way is **Colab Secrets** — your key is stored in your Google account, never
inside this notebook, and it works in every notebook you open from now on.

Don't have a key yet? Get one free at **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)**.

In [ ]:
# ============================================
# STEP 2 — API KEY (Colab Secrets first)
# ============================================
key_panel = zc.ApiKeyPanel()
key_panel.display()
display(HTML(zc.api_key_migration_notice()))

## Step 2.5: Connect Google Drive (strongly recommended) ☁️

A spreadsheet with hundreds of rows takes a while. With Drive connected the results are
saved as the run progresses, so an interrupted session doesn't cost you the whole job —
and starting it again picks up where it stopped.

In [ ]:
# ============================================
# STEP 2.5 — GOOGLE DRIVE
# ============================================
drive_status = widgets.HTML()

drive_save_enabled = widgets.Checkbox(
    value=True,
    description='Save results to Google Drive as the run progresses',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='460px'),
    disabled=True,
)

drive_folder_input = widgets.Text(
    value=drive.folder_name,
    description='Folder:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px'),
    disabled=True,
)


def connect_drive(_button):
    drive_status.value = "<span style='color:#1565c0;'>🔄 Connecting…</span>"
    if drive.mount():
        drive_save_enabled.disabled = False
        drive_folder_input.disabled = False
        # Step 3 may not have been run yet, in which case there is no file
        # picker to refresh -- and that is fine, it will build itself connected.
        if 'file_selector' in globals():
            file_selector.build_drive_tab()
        drive_status.value = (
            "<div style='background:#e8f5e9;padding:12px;border-radius:6px;margin-top:8px;'>"
            "✅ <b>Google Drive connected.</b><br>"
            f"Results will appear in <code>My Drive/{drive.folder_name}/</code><br>"
            "<i>The Drive tab in Step 3 is ready to use.</i></div>"
        )
    else:
        drive_status.value = (
            "<span style='color:#c62828;'>❌ Could not connect. You can still upload "
            "and download files manually.</span>"
        )


def on_folder_change(change):
    drive.folder_name = change['new'].strip() or 'Colab_Summaries'


drive_folder_input.observe(on_folder_change, names='value')

connect_button = widgets.Button(
    description='☁️ Connect Google Drive',
    button_style='primary',
    layout=widgets.Layout(width='230px', height='40px'),
)
connect_button.on_click(connect_drive)

display(connect_button)
display(drive_status)
display(HTML("<br><b>Saving options (available once connected):</b>"))
display(drive_save_enabled)
display(drive_folder_input)

## Step 3: Choose your texts 📁

- **Text files** (`.txt`, `.md`) — each one gets its own summary
- **Spreadsheets** (`.xlsx`) — a *Summary* and a *Keywords* column are added next to your text

Text files produced by the OCR notebook work here directly.

In [ ]:
# ============================================
# STEP 3 — CHOOSE FILES
# ============================================
file_selector = zc.FileSelector(
    dest_dir=FOLDERS['input'],
    extensions=ALL_EXTENSIONS,
    drive=drive,
    icon_for=document_icon,
    what='texts',
    size_hint_mb=100,
)
file_selector.display()

## Step 4: Settings 🎛️

If you are summarising a spreadsheet, press **Read column names** so you can pick which
column holds the text.

In [ ]:
# ============================================
# STEP 4 — SETTINGS
# ============================================
model_dropdown = widgets.Dropdown(
    options=zc.MODEL_CHOICES,
    value=zc.MODEL_FLASH,
    description='Model:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='520px'),
)

model_info = widgets.HTML()


def update_model_info(change):
    model = change['new']
    if 'pro' in model:
        model_info.value = (
            "<div style='background:#e8f5e9;padding:10px;border-radius:5px;margin:5px 0;'>"
            "🎯 <b>Best quality.</b> Worth it for dense or specialised material where the "
            "summary has to be precise.<br>"
            "<b>Needs billing enabled on your Google Cloud project</b> — it is not part of the free tier. "
            "</div>"
        )
    elif 'lite' in model:
        model_info.value = (
            "<div style='background:#fff8e1;padding:10px;border-radius:5px;margin:5px 0;'>"
            "🪶 <b>Fastest and cheapest.</b> A good fit here: summarising is an easier job "
            "than transcribing, so this often holds up well across thousands of rows. "
            "Compare it against Flash on twenty rows before running the whole sheet."
            "</div>"
        )
    else:
        model_info.value = (
            "<div style='background:#e3f2fd;padding:10px;border-radius:5px;margin:5px 0;'>"
            "⚡ <b>Faster and cheaper — the sensible default for summaries.</b> "
            "Recommended when you have a lot of rows to get through."
            "</div>"
        )


model_dropdown.observe(update_model_info, names='value')
update_model_info({'new': model_dropdown.value})

column_dropdown = widgets.Dropdown(
    options=['OCR'],
    value='OCR',
    description='Text column:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='420px'),
)
column_status = widgets.HTML("<i>Press the button to read the column names from your spreadsheet.</i>")


def load_columns(_button):
    sheets = [
        Path(p) for p in file_selector.selected
        if Path(p).suffix.lower() in SHEET_EXTENSIONS
    ]
    if not sheets:
        column_status.value = (
            "<span style='color:#ef6c00;'>⚠️ No spreadsheet selected in Step 3. "
            "This setting only matters for .xlsx files.</span>"
        )
        return
    try:
        columns = list(pd.read_excel(sheets[0], nrows=0).columns)
    except Exception as exc:
        column_status.value = f"<span style='color:#c62828;'>❌ Could not read it: {exc}</span>"
        return
    if not columns:
        column_status.value = "<span style='color:#c62828;'>❌ That spreadsheet has no columns.</span>"
        return
    column_dropdown.options = columns
    column_dropdown.value = 'OCR' if 'OCR' in columns else columns[0]
    column_status.value = (
        f"<span style='color:#2e7d32;'>✅ Read {len(columns)} column(s) from "
        f"{sheets[0].name}. Pick the one holding your text.</span>"
    )


columns_button = widgets.Button(
    description='🔄 Read column names',
    button_style='info',
    layout=widgets.Layout(width='200px'),
)
columns_button.on_click(load_columns)

use_custom_prompt = widgets.Checkbox(
    value=False,
    description='Write my own instructions instead',
    style={'description_width': 'initial'},
)
custom_prompt_text = widgets.Textarea(
    placeholder=(
        'For example: Summarise in three bullet points, focusing on the '
        'religious institutions mentioned.'
    ),
    layout=widgets.Layout(width='520px', height='140px'),
    disabled=True,
)


def toggle_custom(change):
    custom_prompt_text.disabled = not change['new']


use_custom_prompt.observe(toggle_custom, names='value')

save_every_slider = widgets.IntSlider(
    value=10, min=1, max=50, step=1,
    description='Save every:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='420px'),
)

restart_checkbox = widgets.Checkbox(
    value=False,
    description='Start again from scratch (ignore an interrupted earlier run)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='520px'),
)

display(HTML("<h3>🤖 Model</h3>"))
display(model_dropdown)
display(model_info)

display(HTML("<h3>📊 Spreadsheets</h3>"))
display(widgets.HBox([column_dropdown, columns_button]))
display(column_status)

display(HTML("<h3>📝 Instructions</h3>"))
display(use_custom_prompt)
display(custom_prompt_text)
display(HTML(
    "<i>The default asks for a few sentences plus 5–10 keywords. You can also edit "
    "<code>prompts/summary_prompt.md</code> in the file browser on the left.</i>"
))

display(HTML("<h3>💾 Long runs</h3>"))
display(save_every_slider)
display(HTML("<i>How often the spreadsheet is written back to disk, in rows.</i>"))
display(restart_checkbox)

## Step 5: Summarise 🚀

Press the button and leave the tab open. Spreadsheets are saved regularly as the run
progresses, and an interrupted run continues where it left off next time.

In [ ]:
# ============================================
# STEP 5 — SUMMARISATION
# ============================================
summary_output = widgets.Output()
summary_results = {}

KEYWORD_MARKERS = ["Keywords:", "Mots-clés:", "Key words:", "Tags:", "الكلمات المفتاحية:"]


def split_summary_and_keywords(text):
    """Separate the summary from a trailing keyword list, in plain text.

    Only used as a fallback now that replies come back as JSON — but a
    truncated reply is not valid JSON, and salvaging a partial summary beats
    discarding it.
    """
    if not text:
        return ("", "")
    for marker in KEYWORD_MARKERS:
        if marker in text:
            head, _, tail = text.partition(marker)
            keywords = re.sub(r'[\n,;]+', '|', tail.strip())
            keywords = re.sub(r'\s*\|\s*', ' | ', keywords).strip(' |')
            # Stop at the first marker found. Continuing would let a later
            # marker overwrite a correct split with a worse one.
            return (head.strip(), keywords)
    return (text.strip(), "")


def salvage_truncated_json(raw):
    """Recover the summary from a JSON reply that was cut off mid-flight."""
    match = re.search(r'"summary"\s*:\s*"((?:[^"\\]|\\.)*)', raw)
    if not match:
        return None
    try:
        return json.loads(f'"{match.group(1)}"')
    except ValueError:
        return match.group(1)


def parse_summary_response(raw):
    """Read the structured reply, falling back to text parsing if need be."""
    if not raw:
        return ("", "")
    try:
        data = json.loads(raw)
    except (ValueError, TypeError):
        # Most likely a truncated reply. Rescue the summary rather than
        # dropping a fragment of raw JSON into a spreadsheet cell.
        salvaged = salvage_truncated_json(raw)
        if salvaged:
            return (salvaged.strip(), "")
        return split_summary_and_keywords(raw)
    if not isinstance(data, dict):
        return split_summary_and_keywords(raw)

    summary = str(data.get('summary') or '').strip()
    keywords = data.get('keywords') or ''
    if isinstance(keywords, (list, tuple)):
        keywords = " | ".join(str(k).strip() for k in keywords if str(k).strip())
    return (summary, str(keywords).strip())


def load_prompt_template():
    if use_custom_prompt.value and custom_prompt_text.value.strip():
        template = custom_prompt_text.value.strip()
        if "{text}" not in template:
            template += "\n\n{text}"
        return template, "Custom instructions"
    path = Path(FOLDERS['prompts'], 'summary_prompt.md')
    return path.read_text(encoding='utf-8'), 'summary_prompt.md'


def summarise(client, model, config, template, text, label, tokens):
    if not text or not str(text).strip():
        return (None, 'empty')
    prompt = template.replace("{text}", str(text))
    result, status = zc.send_text(
        client, model, config, prompt, label=label, verbose=False, usage_sink=tokens
    )
    if not zc.status_is_usable(status):
        return (None, status)
    zc.warn_if_truncated(status, label)
    return (result.strip(), status)


def save_sheet(frame, output_path, mirror_path):
    frame.to_excel(output_path, index=False)
    if mirror_path:
        try:
            shutil.copy2(output_path, mirror_path)
        except Exception as exc:
            print(f"   ⚠️ Could not copy to Drive (continuing locally): {exc}")


def process_sheet(client, model, config, template, source, tokens):
    """Summarise every row of a spreadsheet, saving as it goes."""
    column = column_dropdown.value
    output_name = f"Summarised_{Path(source).stem}.xlsx"
    output_path = Path(FOLDERS['results']) / output_name
    mirror_path = (
        drive.path_for(output_name)
        if drive.mounted and drive_save_enabled.value else None
    )

    # Reading back a previous partial run is what makes an interrupted job
    # resumable rather than wasted.
    resuming = output_path.exists() and not restart_checkbox.value
    frame = pd.read_excel(output_path if resuming else source)

    if column not in frame.columns:
        available = ", ".join(str(c) for c in frame.columns)
        print(f"   ❌ No column called '{column}'. Available columns: {available}")
        print("      Pick the right one in Step 4 and run again.")
        return None, 0

    for extra in ('Summary', 'Keywords'):
        if extra not in frame.columns:
            frame[extra] = ''
    frame['Summary'] = frame['Summary'].astype('object')
    frame['Keywords'] = frame['Keywords'].astype('object')

    already_done = int(frame['Summary'].astype(str).str.strip().ne('').sum())
    if resuming and already_done:
        print(f"   ↩️ Continuing an earlier run — {already_done} row(s) already summarised")

    total = len(frame)
    done = 0
    skipped = 0
    failed = 0
    save_every = save_every_slider.value

    for position, index in enumerate(frame.index, start=1):
        existing = str(frame.at[index, 'Summary']).strip()
        if existing and existing.lower() != 'nan':
            continue

        source_text = frame.at[index, column]
        if pd.isna(source_text) or not str(source_text).strip():
            skipped += 1
            continue
        if str(source_text).startswith(('[ERROR:', '[SKIPPED:')):
            skipped += 1
            continue

        text, status = summarise(
            client, model, config, template, source_text,
            f"row {position}", tokens,
        )
        if text is None:
            failed += 1
            if status != 'empty':
                print(f"   ⚠️ Row {position}: {status}")
            continue

        summary, keywords = parse_summary_response(text)
        frame.at[index, 'Summary'] = summary
        frame.at[index, 'Keywords'] = keywords
        done += 1

        if done % save_every == 0:
            save_sheet(frame, output_path, mirror_path)
            print(f"   💾 {done} summarised (of {total} rows) — saved")

    save_sheet(frame, output_path, mirror_path)
    print(f"   ✅ {done} row(s) summarised, {skipped} skipped (empty), {failed} failed")
    return output_path, done


def process_text_file(client, model, config, template, source, tokens):
    """Summarise one plain-text file."""
    text = Path(source).read_text(encoding='utf-8', errors='replace')
    result, status = summarise(
        client, model, config, template, text, Path(source).name, tokens
    )
    if result is None:
        print(f"   ⚠️ Could not summarise: {status}")
        return None

    summary, keywords = parse_summary_response(result)
    output_name = f"Summary_{Path(source).stem}.txt"
    output_path = Path(FOLDERS['results']) / output_name
    body = summary + (f"\n\nKeywords: {keywords}" if keywords else "") + "\n"
    output_path.write_text(body, encoding='utf-8')

    if drive.mounted and drive_save_enabled.value:
        mirror = drive.path_for(output_name)
        if mirror:
            try:
                shutil.copy2(output_path, mirror)
            except Exception as exc:
                print(f"   ⚠️ Could not copy to Drive: {exc}")

    print("   ✅ Summary written")
    return output_path


def run_summarisation(_button):
    global summary_results
    summary_results = {}

    with summary_output:
        clear_output()

        api_key = key_panel.get()
        if not api_key:
            print(zc.key_help_message())
            return

        chosen = [Path(p) for p in file_selector.selected]
        if not chosen:
            print("❌ No files chosen. Go back to Step 3.")
            return

        sheets = [p for p in chosen if p.suffix.lower() in SHEET_EXTENSIONS]
        texts = [p for p in chosen if p.suffix.lower() in TEXT_EXTENSIONS]

        model = model_dropdown.value
        template, template_name = load_prompt_template()

        client = zc.make_client(api_key)
        # Use whatever came back: if the chosen alias has been retired, this
        # falls back to one that still exists rather than dead-ending.
        model, model_note = zc.resolve_model(client, model, fallback=zc.MODEL_FALLBACK)
        if not model:
            print(model_note)
            return

        config = zc.build_config(model_id=model, response_schema=SUMMARY_SCHEMA)
        tokens = []

        print(f"🤖 Model: {model_note}")
        print(f"📝 Instructions: {template_name}")
        if sheets:
            print(f"📊 Text column: {column_dropdown.value}")
            print(f"💾 Saving every {save_every_slider.value} row(s)")
        if drive.mounted and drive_save_enabled.value:
            print(f"☁️ Saving to Drive: My Drive/{drive.folder_name}/")
        print("🔄 Auto-retry: 5 attempts on rate limits and server errors")
        print("=" * 55)

        for source in sheets:
            print(f"\n📊 Spreadsheet: {source.name}")
            print("-" * 45)
            try:
                path, done = process_sheet(client, model, config, template, source, tokens)
                if path:
                    summary_results[path.name] = {'path': str(path)}
                    print(f"   💾 Saved: {path}")
            except Exception as exc:
                print(f"   ❌ Could not process {source.name}: {exc}")

        for source in texts:
            print(f"\n📝 Text file: {source.name}")
            print("-" * 45)
            try:
                path = process_text_file(client, model, config, template, source, tokens)
                if path:
                    summary_results[path.name] = {'path': str(path)}
                    print(f"   💾 Saved: {path}")
            except Exception as exc:
                print(f"   ❌ Could not process {source.name}: {exc}")

        print("\n" + "=" * 55)
        print("🎉 FINISHED")
        print(f"   Files produced: {len(summary_results)}")
        if tokens:
            print(f"   🔢 Tokens used: {sum(tokens):,} across {len(tokens)} request(s)")
        print(f"   📁 Local folder: {FOLDERS['results']}/")
        if drive.mounted and drive_save_enabled.value:
            print(f"   ☁️ Drive folder: My Drive/{drive.folder_name}/")
        print("\n👇 Download your results in the next step.")


summary_button = widgets.Button(
    description='🚀 Start summarising',
    button_style='success',
    layout=widgets.Layout(width='220px', height='50px'),
)
summary_button.on_click(run_summarisation)

display(summary_button)
display(HTML("<br>"))
display(summary_output)

## Step 6: Download your results 📥

The ZIP is the reliable option — browsers block long runs of separate downloads.

In [ ]:
# ============================================
# STEP 6 — DOWNLOAD
# ============================================
from google.colab import files as colab_files

download_output = widgets.Output()


def download_zip(_button):
    with download_output:
        clear_output()
        folder = Path(FOLDERS['results'])
        found = [f for f in folder.glob('*') if f.is_file()]
        if not found:
            print("❌ Nothing to download yet — run Step 5 first.")
            return
        print(f"📦 Packing {len(found)} file(s)…")
        shutil.make_archive('summary_results', 'zip', folder)
        colab_files.download('summary_results.zip')
        print("✅ Download started — check your browser's downloads.")


def download_individually(_button):
    with download_output:
        clear_output()
        found = sorted(f for f in Path(FOLDERS['results']).glob('*') if f.is_file())
        if not found:
            print("❌ Nothing to download yet — run Step 5 first.")
            return
        if len(found) > 5:
            print(f"⚠️ {len(found)} files — browsers usually block this many. "
                  "Use the ZIP button instead if some are missing.\n")
        for path in found:
            print(f"   {path.name}")
            try:
                colab_files.download(str(path))
            except Exception as exc:
                print(f"   ⚠️ {path.name}: {exc}")
        print("\n✅ Downloads started.")


zip_button = widgets.Button(description='📦 Download all as ZIP', button_style='success',
                            layout=widgets.Layout(width='240px', height='40px'))
zip_button.on_click(download_zip)

each_button = widgets.Button(description='📄 Download one by one', button_style='',
                             layout=widgets.Layout(width='240px', height='40px'))
each_button.on_click(download_individually)

display(widgets.HBox([zip_button, each_button]))
display(HTML(f"<br><i>Everything is also kept in <code>{FOLDERS['results']}/</code></i>"))
display(download_output)

## Step 7 (optional): Tidy up 🧹

Colab throws everything away when the session ends, so this is only useful if you are
running out of space or want a clean slate. Anything already copied to Drive is safe.

**Careful:** deleting the results also deletes the partial spreadsheet an interrupted run
would otherwise continue from.

In [ ]:
# ============================================
# STEP 7 — CLEANUP
# ============================================
cleanup_output = widgets.Output()


def clear_inputs(_button):
    with cleanup_output:
        clear_output()
        count = zc.clear_folder(FOLDERS['input'])
        file_selector.selected = []
        print(f"🧹 Uploaded files: {count} file(s) deleted")


def clear_results(_button):
    global summary_results
    with cleanup_output:
        clear_output()
        count = zc.clear_folder(FOLDERS['results'])
        summary_results = {}
        print(f"🧹 Results: {count} file(s) deleted")


def clear_everything(_button):
    global summary_results
    with cleanup_output:
        clear_output()
        count = zc.clear_folder(FOLDERS['input']) + zc.clear_folder(FOLDERS['results'])
        file_selector.selected = []
        summary_results = {}
        print(f"🧹 {count} file(s) deleted. Prompts kept.")


def show_status(_button):
    with cleanup_output:
        clear_output()
        zc.folder_report(FOLDERS)


btn_status = widgets.Button(description='📊 What is here?', button_style='info',
                            layout=widgets.Layout(width='170px'))
btn_status.on_click(show_status)

btn_inputs = widgets.Button(description='🗑️ Uploaded files', button_style='warning',
                            layout=widgets.Layout(width='170px'))
btn_inputs.on_click(clear_inputs)

btn_results = widgets.Button(description='🗑️ Results', button_style='warning',
                             layout=widgets.Layout(width='170px'))
btn_results.on_click(clear_results)

btn_all = widgets.Button(description='🗑️ Everything', button_style='danger',
                         layout=widgets.Layout(width='170px'))
btn_all.on_click(clear_everything)

display(HTML("<b>Safe:</b>"))
display(btn_status)
display(HTML("<br><b>⚠️ These delete your files:</b>"))
display(widgets.HBox([btn_inputs, btn_results]))
display(HTML("<br><b>🔴 Everything at once:</b>"))
display(btn_all)
display(HTML("<i>Prompt templates are never deleted.</i>"))
display(HTML("<br>"))
display(cleanup_output)

---

## ℹ️ Help

### Spreadsheets

Your file keeps all its original columns. Two are added: **Summary** and **Keywords**.
Rows that already have a summary are left alone, so you can stop a run and start it again
without paying twice for the same rows.

Press **Read column names** in Step 4 to choose which column holds the text. It does not
have to be called `OCR` — that is only the default because it is what the OCR notebook
produces.

### Interrupted runs

Results are written to `results/Summarised_<your file>.xlsx` every few rows. Starting the
same job again continues from that file. To ignore it and start over, tick *Start again
from scratch* in Step 4.

### Working from the OCR notebook

Text files from the OCR notebook can be summarised here directly — the `_ocr.txt` files
contain only the transcribed text. (Their `.info.txt` companions are just provenance
records; there is no need to summarise those.)

### Common problems

**"No API key found"** — add a Colab Secret named `GEMINI_API_KEY` (🔑 icon, left sidebar)
and switch *Notebook access* on, then press *Check Secrets again* in Step 2.

**A key that used to work now fails** — Google is retiring old-style keys, and all of them
stop working in September 2026. Create a fresh one at
[aistudio.google.com/apikey](https://aistudio.google.com/apikey).

**"No column called …"** — press *Read column names* in Step 4 and pick from the list.

**Rate limit errors** — the notebook retries automatically. If a large spreadsheet keeps
failing, wait a few minutes and start it again; it will resume.

### Privacy

On the free tier Google may use what you send to improve its products, and its terms
permit human review. With billing enabled, it does not. See the
[Gemini API terms](https://ai.google.dev/gemini-api/terms).

---

### About

**ZMO AI Pipelines**, created by [Frédérick Madore](https://www.frederickmadore.com/).

Part of the [Leibniz-Zentrum Moderner Orient (ZMO)](https://www.zmo.de/) research tools.